<h2 style="color:#4A148C; margin-bottom:6px;">
  07 – RoBERTa Base with Data Augmentation for Multi-Label Emotion Classification
</h2>

<p style="font-size:16px; margin-top:4px;">
  <strong>Overview:</strong>
  This notebook fine-tunes a <strong>RoBERTa Base</strong> model for
  multi-label emotion classification using an <strong>augmented
  training set</strong>. The augmentation pipeline combines
  <em>synonym-based</em> word replacement (WordNet) with
  <em>contextual word substitutions</em> generated by a
  <strong>BERT-based language model</strong> via the <code>nlpaug</code>
  library.
</p>

<p style="font-size:16px;">
  <strong>Objective:</strong>
  To investigate whether augmenting the original training data with
  semantically varied sentences can improve the Macro F1-score for
  predicting the five emotions
  (<em>anger, fear, joy, sadness, surprise</em>) compared to the
  non-augmented RoBERTa baseline.
</p>

<p style="font-size:16px;">
  The final model is trained on the concatenation of original and
  augmented samples, validated on a held-out split, and the best
  checkpoint (based on validation Macro F1) is later reused in a
  separate inference notebook to generate Kaggle submissions.
</p>


In [2]:
#wandb login check to make sure everything goes smooth

from kaggle_secrets import UserSecretsClient
import wandb

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=api_key)

print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sharmilam-official (sharmila-m) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful!


In [3]:
pip install nlpaug 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 10.7 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
# 07a – RoBERTa Base with Data Augmentation (Training)

import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

import nlpaug.augmenter.word as naw

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

import wandb
import kagglehub

# -------------------------
# 0. Config
# -------------------------
LABELS = ['anger','fear','joy','sadness','surprise']
MAX_LEN = 80
BATCH_SIZE = 16
EPOCHS = 8
PATIENCE = 3
LR = 2e-5
RANDOM_STATE = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# -------------------------
# 1. WandB Init
# -------------------------
wandb.init(
    project="24ds2000048-t32025",
    entity="sharmila-m",
    name="07a_roberta_aug_bertctx",
    config={
        "model_name": "roberta-base",
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "patience": PATIENCE,
        "augmentation": "synonym + BERT contextual",
        "random_state": RANDOM_STATE
    }
)
config = wandb.config

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# -------------------------
# 2. Load Data
# -------------------------
train_df = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/train.csv')
test_df  = pd.read_csv('/kaggle/input/2025-sep-dl-gen-ai-project/test.csv')

print("Original train size:", len(train_df))

# -------------------------
# 3. Data Augmentation
#    Synonym + BERT-based contextual augmentation
# -------------------------
syn_aug = naw.SynonymAug(aug_src='wordnet', aug_max=2)

ctx_aug = naw.ContextualWordEmbsAug(
    model_path='bert-base-uncased',  # BERT used for augmentation only
    action="substitute",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

augmented_texts = []
augmented_labels = []

for idx, row in tqdm(train_df.iterrows(),
                     total=len(train_df),
                     desc='Augmenting data'):
    orig_text = row['text']

    # 1) Synonym-based augmentation
    try:
        syn_text = syn_aug.augment(orig_text)
    except Exception:
        syn_text = orig_text

    # 2) Contextual BERT-based augmentation
    try:
        bert_text = ctx_aug.augment(orig_text)
    except Exception:
        bert_text = orig_text

    augmented_texts.extend([syn_text, bert_text])
    augmented_labels.extend([row[LABELS].values] * 2)

aug_df = pd.DataFrame({'text': augmented_texts})
aug_df[LABELS] = pd.DataFrame(augmented_labels, index=aug_df.index)

train_aug_df = pd.concat([train_df, aug_df], ignore_index=True)
train_aug_df = train_aug_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Augmented train size:", len(train_aug_df))

# -------------------------
# 4. Tokenization & Dataset (RoBERTa)
# -------------------------
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, mode='train'):
        self.df = df
        self.tokenizer = tokenizer
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['text'])
        encoding = self.tokenizer(
            text,
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.mode == 'train':
            labels = torch.tensor(
                self.df.iloc[idx][LABELS].values.astype('float32')
            )
            item['labels'] = labels
        return item

train_df_split, val_df_split = train_test_split(
    train_aug_df,
    test_size=0.1,
    random_state=RANDOM_STATE
)

train_dataset = EmotionDataset(train_df_split, tokenizer, mode='train')
val_dataset   = EmotionDataset(val_df_split, tokenizer, mode='train')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

# -------------------------
# 5. RoBERTa Model
# -------------------------
class RobertaForMultiLabel(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits  # raw logits

model = RobertaForMultiLabel(len(LABELS)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

# -------------------------
# 6. Training & Validation with Early Stopping
# -------------------------
best_val_f1 = -1.0
wait = 0
best_model_path = "roberta_aug_best.pt"

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Train"):
        input_ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attn)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_labels, val_preds = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} - Val"):
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attn)
            probs = torch.sigmoid(logits).cpu().numpy()

            val_preds.append(probs)
            val_labels.append(labels.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_labels = np.vstack(val_labels)
    pred_bin = (val_preds > 0.5).astype(int)

    val_f1_macro = f1_score(val_labels, pred_bin, average='macro')
    val_accuracy = accuracy_score(val_labels, pred_bin)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Macro F1: {val_f1_macro:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_f1_macro": val_f1_macro,
        "val_accuracy": val_accuracy
    })

    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        wait = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

print("Best validation Macro F1:", best_val_f1)
wandb.log({"best_val_f1_macro": best_val_f1})

# -------------------------
# 7. Upload Best Model to KaggleHub
# -------------------------
KAGGLE_USERNAME = "sharmilamamani"   
MODEL = "emotion-roberta-aug"
FRAMEWORK = "pytorch"
VARIATION = "roberta-bertaug-v1"

handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

try:
    print("Uploading best RoBERTa+Aug model to KaggleHub...")
    repo = kagglehub.model_upload(
        handle,
        best_model_path,
        version_notes="RoBERTa base with synonym + BERT contextual augmentation"
    )
    print("KaggleHub upload successful.")
    print("Model handle:", handle)
except Exception as e:
    print("KaggleHub upload failed:")
    print(type(e).__name__, ":", e)

wandb.finish()


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
2025-11-29 13:34:02.359690: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764423242.550857      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764423242.600729      47 cuda_blas.cc:1418] Unab

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Epoch 8/8 | Train Loss: 0.0336 | Val Macro F1: 0.8837 | Val Acc: 0.7897
Best validation Macro F1: 0.8836793016934232
Uploading best RoBERTa+Aug model to KaggleHub...
Uploading Model https://www.kaggle.com/models/sharmilamamani/emotion-roberta-aug/pytorch/roberta-bertaug-v1 ...
Model 'emotion-roberta-aug' does not exist or access is forbidden for user 'sharmilamamani'. Creating or handling Model...
Model 'emotion-roberta-aug' Created.
Starting upload for file roberta_aug_best.pt


Uploading: 100%|██████████| 499M/499M [00:06<00:00, 79.3MB/s] 

Upload successful: roberta_aug_best.pt (476MB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/sharmilamamani/emotion-roberta-aug/pytorch/roberta-bertaug-v1
KaggleHub upload successful.
Model handle: sharmilamamani/emotion-roberta-aug/pytorch/roberta-bertaug-v1


best_val_f1_macro,▁
epoch,▁▂▃▄▅▆▇█
train_loss,█▅▄▂▂▁▁▁
val_accuracy,▁▄▆▇████
val_f1_macro,▁▅▇▇████
best_val_f1_macro,0.88368
epoch,8
train_loss,0.03361
val_accuracy,0.78965
val_f1_macro,0.88368


<hr/>

<h3 style="color:#D35400;">
  Results & Runtime Summary
</h3>

<p style="font-size:16px;">
  <strong>Model:</strong>
  RoBERTa Base with synonym + BERT-based contextual data augmentation
  for multi-label emotion classification
</p>

<p style="font-size:16px;">
  <strong>Final Training Epoch:</strong> 8<br/>
  Train Loss: <strong>0.03361</strong>
</p>

<p style="font-size:16px;">
  <strong>Validation Metrics (Internal):</strong><br/>
  Macro F1 Score: <strong>0.88368</strong><br/>
  Accuracy: <strong>0.78965</strong>
</p>

<p style="font-size:16px;">
  <strong>Kaggle Public Leaderboard Score:</strong><br/>
  Macro F1 Score: <strong>0.851</strong>
</p>

<p style="font-size:16px;">
  <strong>Runtime Environment:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li><strong>Platform:</strong> Kaggle Notebook</li>
  <li><strong>Hardware:</strong> GPU (NVIDIA T4 &times; 2)</li>
  <li><strong>Estimated Runtime:</strong> ~45–55 minutes (including augmentation, training, and validation)</li>
  <li><strong>Frameworks:</strong> PyTorch, HuggingFace Transformers, nlpaug, scikit-learn, pandas</li>
</ul>

<p style="font-size:16px;">
  <strong>Observations:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li>
    The augmented RoBERTa model achieves a high internal validation
    Macro F1 score of <strong>0.8837</strong>, indicating strong
    performance on the held-out validation split.
  </li>
  <li>
    The Kaggle leaderboard score (<strong>0.851</strong>) is slightly
    lower than the internal validation score but still represents a clear
    improvement over the non-augmented RoBERTa baseline, showing that
    data augmentation improves generalization.
  </li>
  <li>
    The combination of lexical synonym replacement and BERT-based
    contextual substitutions increases linguistic diversity in the
    training data while preserving the underlying emotion labels, making
    this the best-performing model in the overall pipeline.
  </li>
</ul>
